In [2]:
pip install scipy statsmodels


     ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
     ---------------------------------------- 0.1/9.8 MB 1.5 MB/s eta 0:00:07
      --------------------------------------- 0.2/9.8 MB 3.0 MB/s eta 0:00:04
     - -------------------------------------- 0.3/9.8 MB 2.3 MB/s eta 0:00:05
     - -------------------------------------- 0.3/9.8 MB 2.1 MB/s eta 0:00:05
     - -------------------------------------- 0.3/9.8 MB 2.1 MB/s eta 0:00:05
     - -------------------------------------- 0.4/9.8 MB 1.3 MB/s eta 0:00:08
     - -------------------------------------- 0.5/9.8 MB 1.4 MB/s eta 0:00:07
     -- ------------------------------------- 0.5/9.8 MB 1.4 MB/s eta 0:00:07
     -- ------------------------------------- 0.5/9.8 MB 1.3 MB/s eta 0:00:07
     -- ------------------------------------- 0.5/9.8 MB 1.3 MB/s eta 0:00:07
     -- ------------------------------------- 0.6/9.8 MB 1.1 MB/s eta 0:00:09
     -- ------------------------------------- 0.6/9.8 MB 1.1 MB/s eta 0


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\15520\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd 
from scipy.stats import levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [2]:
data2 = pd.read_csv('BT2.225092301_insurance_survey.csv', delimiter=";") 
data2.head()

,Age,Gender,Education,Marital Status,Years Employed,Satisfaction*,Premium/Deductible**
0,36,F,Some college,Divorced,4,4,N
1,55,F,Some college,Divorced,2,1,N
2,61,M,Graduate degree,Widowed,26,3,N
3,65,F,Some college,Married,9,4,N
4,53,F,Graduate degree,Married,6,4,N


Chia dữ liệu thành các nhóm dựa trên giá trị cột Education, tạo 1 từ điển gồm các df chứa dữ liệu của từng nhóm

In [ ]:
# Tạo từ điển chứa các DataFrame của từng nhóm
education_groups = {edu: df for edu, df in data2.groupby('Education')}

# Kiểm tra danh sách các nhóm
print("Danh sách các nhóm Education:", list(education_groups.keys()))

# Xem thử một nhóm cụ thể (ví dụ: 'Bachelor')
print("\nDữ liệu nhóm Some college:")
print(education_groups.get('Some college').head())  # Hiển thị 5 dòng đầu


Danh sách các nhóm Education: ['College graduate', 'Graduate degree', 'Some college']

Dữ liệu nhóm Some college:
    Age Gender     Education Marital Status  Years Employed  Satisfaction*   \
0    36      F  Some college       Divorced               4               4   
1    55      F  Some college       Divorced               2               1   
3    65      F  Some college        Married               9               4   
12   38      M  Some college        Married               3               2   
13   27      M  Some college        Married               2               3   

   Premium/Deductible**  
0                     N  
1                     N  
3                     N  
12                    N  
13                    N  
{'College graduate':     Age Gender         Education Marital Status  Years Employed  \
6    28      F  College graduate        Married               4   
7    62      F  College graduate       Divorced               9   
10   57      F  College graduate 

Thưc hiên và in kết quả kiểm định Leneve với mức ý nghĩa 0.05. Bao gồm các bước đặt ra giả thuyết bài toán , in kết quả, kết luận

H0: Phương sai của mức độ hài lòng là như nhau giữa các nhóm trình độ học vấn.
H1: Có sự khác biệt về phương sai giữa các nhóm trình độ học vấn.

In [14]:
# Lọc dữ liệu: Chỉ lấy các cột liên quan
education_groups = [group['Satisfaction* '].dropna().values for _, group in data2.groupby('Education')]
print(education_groups)

[array([5, 3, 5, 3, 3, 3, 3, 4, 2]), array([3, 4, 5, 5, 5, 4, 5, 5]), array([4, 1, 4, 2, 3, 4, 4])]


In [19]:
import numpy as np
import pandas as pd
from scipy.stats import levene

# Giả sử data2 là DataFrame chứa cột 'Education' và 'Satisfaction'
education_groups = {edu: df['Satisfaction* '].dropna().values for edu, df in data2.groupby('Education')}

# Xác định số nhóm k
k = len(education_groups)
print(f"Số nhóm (k): {k}")

# Tổng số quan sát N
N = sum(len(group) for group in education_groups.values())
print(f"Tổng số quan sát (N): {N}")

# Số quan sát trong từng nhóm Ni
Ni = {edu: len(group) for edu, group in education_groups.items()}
print(f"Số quan sát trong từng nhóm (Ni): {Ni}")

# Tạo biến đổi Z_ij = |X_ij - X_i.|
Z = {}  # Dictionary lưu các giá trị Z_ij của từng nhóm
for edu, group in education_groups.items():
    mean_i = np.mean(group)  # Trung bình nhóm i
    Z[edu] = np.abs(group - mean_i)  # Áp dụng công thức Z_ij = |X_ij - X_i.|

# Trung bình Z trong từng nhóm Z_i.
Zi_mean = {edu: np.mean(z) for edu, z in Z.items()}
print(f"Trung bình Z trong từng nhóm (Zi_mean): {Zi_mean}")

# Trung bình tổng thể Z..
Z_overall = np.mean([z for group in Z.values() for z in group])
print(f"Trung bình tổng thể Z..: {Z_overall}")

# Tính toán W theo công thức Levene
numerator = sum(Ni[edu] * (Zi_mean[edu] - Z_overall) ** 2 for edu in education_groups.keys())
denominator = sum(sum((z - Zi_mean[edu]) ** 2 for z in Z[edu]) for edu in education_groups.keys())

W = ((N - k) / (k - 1)) * (numerator / denominator)
print(f"Giá trị thống kê Levene (W): {W}")

# So sánh với scipy.stats.levene để kiểm chứng
stat, p_value = levene(*education_groups.values())
print(f"Thống kê Levene từ scipy: {stat:.4f}, P-value: {p_value:.4f}")


Số nhóm (k): 3
Tổng số quan sát (N): 24
Số quan sát trong từng nhóm (Ni): {'College graduate': 9, 'Graduate degree': 8, 'Some college': 7}
Trung bình Z trong từng nhóm (Zi_mean): {'College graduate': np.float64(0.8148148148148149), 'Graduate degree': np.float64(0.625), 'Some college': np.float64(0.979591836734694)}
Trung bình tổng thể Z..: 0.7996031746031745
Giá trị thống kê Levene (W): 0.9433580072525426
Thống kê Levene từ scipy: 0.2652, P-value: 0.7696


fcrit với bậc tự do(k-1, n-k) với mức ý nghĩa alpha 0.05:
f(2,21)0.05 = 3.4668
Ta có W < fcrit => Chấp nhận H0
Phương sai của mức độ hài lòng là đồng nhất giữa các nhóm trình độ học vấn

Thưc hiên kiểm định ANOVA one way, Bao gồm các bước đặt ra giả thuyết bài toán , in kết quả, kết luận, sử dụng f_oneway()


H0: Không có sự khác biệt về mức độ hài lòng giữa các nhóm trình độ học vấn.
H1: Có ít nhất một nhóm có giá trị trung bình khác biệt đáng kể.

In [21]:
# Tính tổng số quan sát và trung bình tổng thể
N = sum(len(group) for group in education_groups.values())  # Tổng số quan sát
X_bar = np.mean(np.concatenate(list(education_groups.values())))  # Trung bình tổng thể
k = len(education_groups)  # Số nhóm

# 📌 Tính SST (Tổng phương sai tổng thể)
SST = sum((X_ij - X_bar)**2 for group in education_groups.values() for X_ij in group)

# 📌 Tính SSB (Tổng phương sai giữa các nhóm)
SSB = sum(len(group) * (np.mean(group) - X_bar)**2 for group in education_groups.values())

# 📌 Tính SSW (Tổng phương sai trong nhóm)
SSW = sum((X_ij - np.mean(group))**2 for group in education_groups.values() for X_ij in group)

# 📌 Tính bậc tự do
df_B = k - 1  # Bậc tự do giữa nhóm
df_W = N - k  # Bậc tự do trong nhóm

# 📌 Tính MSB và MSW
MSB = SSB / df_B
MSW = SSW / df_W

# 📌 Tính F-statistic
F_stat = MSB / MSW

# 📌 Tính P-value
p_value = 1 - stats.f.cdf(F_stat, df_B, df_W)

# 📌 In kết quả chi tiết
print(f"Tổng số quan sát (N): {N}")
print(f"Số nhóm (k): {k}")
print(f"Trung bình tổng thể (X̄): {X_bar:.4f}")
print(f"SST (Tổng phương sai tổng thể): {SST:.4f}")
print(f"SSB (Tổng phương sai giữa nhóm): {SSB:.4f}")
print(f"SSW (Tổng phương sai trong nhóm): {SSW:.4f}")
print(f"Bậc tự do giữa nhóm (df_B): {df_B}")
print(f"Bậc tự do trong nhóm (df_W): {df_W}")
print(f"MSB (Phương sai giữa nhóm): {MSB:.4f}")
print(f"MSW (Phương sai trong nhóm): {MSW:.4f}")
print(f"F-statistic: {F_stat:.4f}")
print(f"P-value: {p_value:.4f}")


Tổng số quan sát (N): 24
Số nhóm (k): 3
Trung bình tổng thể (X̄): 3.7083
SST (Tổng phương sai tổng thể): 28.9583
SSB (Tổng phương sai giữa nhóm): 7.8790
SSW (Tổng phương sai trong nhóm): 21.0794
Bậc tự do giữa nhóm (df_B): 2
Bậc tự do trong nhóm (df_W): 21
MSB (Phương sai giữa nhóm): 3.9395
MSW (Phương sai trong nhóm): 1.0038
F-statistic: 3.9247
P-value: 0.0356


fcrit với bậc tự do(k-1, n-k) với mức ý nghĩa alpha 0.05:
f(2,21)0.05 = 3.4668
Ta có Fstat > fcrit => Bác bỏ H0
Tồn tại sự khác biệt giữa các trinh độ giáo dục vê mức độ hài lòng

In [31]:
from scipy.stats import studentized_range

# Định nghĩa tham số
alpha = 0.05  # Mức ý nghĩa
k = 3  # Số nhóm
df_w = 21  # Bậc tự do trong nhóm

# Tính giá trị tới hạn
q_critical = studentized_range.ppf(1 - alpha, k, df_w)

# In kết quả
print(f"Giá trị tới hạn q({alpha}, {k}, {df_w}): {q_critical:.4f}")


Giá trị tới hạn q(0.05, 3, 21): 3.5646


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import tukey_hsd

# Chuyển đổi từng nhóm thành numpy array và loại bỏ NaN
group_names = list(education_groups.keys())
group_values = [pd.Series(education_groups[name]).dropna().values for name in group_names]

# Tạo danh sách tất cả các cặp nhóm cần so sánh
pairwise_comparisons = list(combinations(range(len(group_names)), 2))


print("📌 Kết quả kiểm định Tukey-Kramer từng cặp nhóm:")
for i, j in pairwise_comparisons:
    ni, nj = len(group_values[i]), len(group_values[j])
    mean_i, mean_j = np.mean(group_values[i]), np.mean(group_values[j])
    
    # Tính toán giá trị T
    T = q_critical * np.sqrt(MSW / 2 * (1/ni + 1/nj))
    
    # Chênh lệch trung bình giữa hai nhóm
    D = abs(mean_i - mean_j)
    
    # Kiểm định giả thuyết
    conclusion = "Bác bỏ H0 (Có sự khác biệt)" if D > np.max(T) else "Chấp nhận H0 (Không có sự khác biệt)"
    
    # In kết quả
    print(f"\n🟢 Cặp nhóm: {group_names[i]} vs {group_names[j]}")
    print(f"  - Trung bình: {mean_i:.4f} vs {mean_j:.4f}")
    print(f"  - Số lượng mẫu: {ni} vs {nj}")
    print(f"  - Giá trị T: {T:.4f}")A
    print(f"  - Chênh lệch D: {D:.4f}")
    print(f"  - Kết luận: {conclusion}")


📌 Kết quả kiểm định Tukey-Kramer từng cặp nhóm:

🟢 Cặp nhóm: College graduate vs Graduate degree
  - Trung bình: 3.4444 vs 4.5000
  - Số lượng mẫu: 9 vs 8
  - Giá trị T: 1.2271
  - Chênh lệch D: 1.0556
  - Kết luận: Chấp nhận H0 (Không có sự khác biệt)

🟢 Cặp nhóm: College graduate vs Some college
  - Trung bình: 3.4444 vs 3.1429
  - Số lượng mẫu: 9 vs 7
  - Giá trị T: 1.2726
  - Chênh lệch D: 0.3016
  - Kết luận: Chấp nhận H0 (Không có sự khác biệt)

🟢 Cặp nhóm: Graduate degree vs Some college
  - Trung bình: 4.5000 vs 3.1429
  - Số lượng mẫu: 8 vs 7
  - Giá trị T: 1.3070
  - Chênh lệch D: 1.3571
  - Kết luận: Bác bỏ H0 (Có sự khác biệt)


Kiểm tra lại bằng hàm pairwise_tukeyhsd()

In [29]:
import numpy as np
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Lấy dữ liệu Satisfaction và Education
satisfaction = data2['Satisfaction* ']
education = data2['Education']

# Thực hiện kiểm định Tukey HSD
tukey_test = pairwise_tukeyhsd(satisfaction, education, alpha=0.05)

# In kết quả
print(tukey_test)


          Multiple Comparison of Means - Tukey HSD, FWER=0.05          
     group1           group2     meandiff p-adj   lower   upper  reject
-----------------------------------------------------------------------
College graduate Graduate degree   1.0556 0.1003 -0.1715  2.2826  False
College graduate    Some college  -0.3016 0.8231 -1.5742  0.9711  False
 Graduate degree    Some college  -1.3571 0.0409 -2.6641 -0.0502   True
-----------------------------------------------------------------------


"College graduate" vs "Graduate degree": p-value = 0.1003 (> 0.05) => Không có sự khác biệt đáng kể về trung bình giữa hai nhóm

 "College graduate" vs "Some college": p-value = 0.8231 (> 0.05) => Không có sự khác biệt đáng kể về trung bình giữa hai nhóm.

 "Graduate degree" vs "Some college": p-value = 0.0409 (< 0.05) => Có sự khác biệt đáng kể về trung bình giữa hai nhóm.